# Data Leakage: Definisi Formal, Taksonomi, Mekanisme, dan Deteksi

## Target Belajar

* Mendefinisikan data leakage secara formal, sesuai definisi resmi yang dipakai literatur dan dokumentasi scikit-learn
* Mengklasifikasikan data leakage ke dalam taksonomi jenis-jenisnya, beserta mekanisme spesifik masing-masing
* Membedakan gejala data leakage dari gejala masalah model lain (overfitting biasa, underfitting)
* Membuktikan secara empiris keberadaan tiap jenis leakage lewat eksperimen kode
* Menerapkan prosedur deteksi dini sebelum model dianggap "berhasil"

# Bagian 1: Penjelasan Konsep

## 1.1 Definisi Formal

* Data leakage: kondisi dimana data / informasi yang tidak tersedian pada saat prediksi dilakukan justru digunakan saat membangun model
* Konsekuensi operasional: Menghasilkan estimasi performa yang terlalu optimis (cross validation), sehingga performa menjadi lebih buruk ketika model dipakai pada data baru yang sesungguhnya, misalnya saat berada di tahap produksi.

## 1.2 Prinsip Umum sebagai Garis Pertahanan Pertama

* Penyebab umum: tidak menjaga pemisahan antara subset data train dan test
* aturan umumnya adalah: jangan pernah memanggil fit pada data test.

## 1.3 Leakage Bukan Fenomena Tunggal — Perlunya Taksonomi

* kebocoran statistik preprocessing (mean, std, kategori) dari test ke proses fitting.
* leakage dapat terjadi melalui mekanisme yang secara struktural berbeda, tidak selalu melibatkan pemanggilan .fit() yang salah secara literal.
* Diperlukan taksonomi eksplisit agar setiap jenis leakage dapat dikenali berdasarkan pola gejalanya masing-masing


## 1.4 Taksonomi Jenis-Jenis Data Leakage

Bentuk utama leakage

| Jenis                        | Definisi Singkat                                                                                                    | Titik Kemunculan dalam Pipeline                                |
| ---------------------------- | ------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------- |
| **Preprocessing leakage**    | Statistik transformasi (mean, std, kategori) dihitung dari seluruh dataset sebelum split                            | Tahap preprocessing (materi 05)                                |
| **Target leakage**           | Fitur mengandung informasi yang merupakan turunan langsung dari target, atau baru tersedia setelah target diketahui | Tahap feature engineering / pengumpulan data                   |
| **Temporal leakage**         | Fitur memakai informasi dari masa depan relatif terhadap titik waktu prediksi seharusnya dibuat                     | Data time series / data dengan dimensi waktu                   |
| **Group leakage**            | Entitas yang sama (pasien, pelanggan, sesi) muncul baik di train maupun test                                        | Tahap split, khususnya pada data dengan struktur pengelompokan |
| **Train-test contamination** | Istilah payung untuk seluruh kasus di mana informasi test “bocor” ke proses training melalui jalur apa pun          | Kategori umum yang mencakup empat jenis di atas                |


![taksonomi_data_leakage.png](../assets/taksonomi_data_leakage.png)

## 1.5 Mekanisme Spesifik Tiap Jenis Leakage

### (a) Preprocessing Leakage

* Mekanisme: statistik (mean, std, kategori unik, dsb.) dihitung menggunakan seluruh dataset — termasuk baris yang akan menjadi test set — sebelum split dilakukan, atau transformer di-fit() pada data test.
* Sudah dibuktikan secara empiris di materi 02 dan 05; tidak diulang di sini, namun akan diperluas di bagian implementasi dengan efek ukuran yang lebih besar.

### (b) Target Leakage

* Mekanisme: fitur input mengandung informasi yang merupakan turunan langsung dari target, atau yang secara kronologis baru tersedia setelah target diketahui.
* Karakteristik pembeda utama: leakage jenis ini tidak bergantung pada kapan split dilakukan — leakage tetap terjadi walau prosedur split dan preprocessing sudah benar, karena masalahnya ada pada pemilihan fitur itu sendiri, bukan pada prosedur train/test.
* Kriteria identifikasi: pertimbangkan urutan kronologis ketersediaan data, bukan semata apakah suatu fitur secara statistik membantu prediksi.
* Contoh ilustratif: memakai fitur "jumlah cicilan yang telah dibayar" untuk memprediksi "apakah nasabah akan gagal bayar" — fitur ini baru terisi setelah status gagal bayar diketahui, sehingga tidak tersedia pada saat prediksi seharusnya dibuat.

(c) Temporal Leakage

* Mekanisme: fitur atau prosedur validasi memakai data dari titik waktu yang berada setelah titik waktu prediksi seharusnya dibuat.
* Bentuk paling umum: melakukan random split (bukan split kronologis) pada data time series — baris dari masa depan berpotensi masuk ke train set, sementara baris dari masa lalu masuk ke test set.
* Relevansi silang: konsep ini merupakan pendalaman langsung dari catatan penting yang sudah diperkenalkan pada materi 3.5 Time Series Modeling — di sini dibahas sebagai kategori formal dalam taksonomi leakage.

### (d) Group Leakage

* Mekanisme: unit observasi yang sebenarnya berasal dari entitas yang sama (pasien, pelanggan, sesi pengguna) terpecah — sebagian barisnya masuk ke train, sebagian masuk ke test.
*  Akibatnya: model tidak diuji kemampuannya untuk generalisasi ke entitas yang benar-benar baru, melainkan hanya "mengenali kembali" entitas yang polanya sudah pernah dilihat sebagian saat training.
* Solusi struktural: pembagian data harus dilakukan pada level grup/entitas, bukan pada level baris individual — dibahas teknis di bagian implementasi menggunakan GroupShuffleSplit

### (e) Train-Test Contamination (Kategori Umum)

Istilah payung yang mencakup seluruh skenario di atas — merujuk pada kondisi umum ketika batas antara train dan test tidak lagi murni, dengan penyebab spesifik apa pun

## 1.6 Prosedur Umum yang Benar vs Prosedur yang Keliru

* Empat jenis leakage di atas memiliki akar penyebab berbeda, namun prinsip pencegahannya bermuara pada satu pola umum yang sama: seluruh keputusan (statistik, pemilihan fitur, pembagian data) harus dibuat seolah-olah test set belum eksis pada saat itu.
* Diagram berikut merangkum kembali secara visual perbandingan dua alur kerja — versi yang keliru (kolom kiri) dan versi yang benar (kolom kanan) — dengan titik percabangan yang menjadi sumber leakage ditandai secara eksplisit pada baris kedua dan ketiga.

![alur_salah_vs_benar_leakage.png](../assets/alur_salah_vs_benar_leakage.png)

## 1.7 Gejala dan Prosedur Deteksi Dini

*  Gejala primer: performa yang sangat tinggi pada data training/cross-validation, namun turun drastis ketika model diuji pada data yang benar-benar baru (data test yang belum pernah disentuh sama sekali dalam proses apa pun, atau data produksi).
* Heuristik kecurigaan umum: skor yang "terlalu bagus untuk jadi kenyataan" (misalnya akurasi > 0.98 pada masalah yang secara domain diketahui sulit) sebaiknya memicu investigasi terhadap leakage terlebih dahulu, sebelum diklaim sebagai keberhasilan model.
* Prosedur verifikasi sistematis untuk masing-masing jenis leakage:

| Jenis Leakage             | Cara Verifikasi                                                                                                               |
| ------------------------- | ----------------------------------------------------------------------------------------------------------------------------- |
| **Preprocessing leakage** | Periksa apakah `.fit()` transformer dipanggil **sebelum atau sesudah split**; audit urutan kode                               |
| **Target leakage**        | Periksa setiap fitur: apakah nilainya akan **diketahui pada saat prediksi sesungguhnya dibuat**, sesuai kronologi dunia nyata |
| **Temporal leakage**      | Periksa apakah split memperhatikan **urutan waktu**; pastikan tidak ada baris “masa depan” di train set                       |
| **Group leakage**         | Periksa apakah ada **identifier entitas** (ID pasien, ID pelanggan) yang muncul di train **dan** test secara bersamaan        |


* Prinsip kehati-hatian tambahan: feature importance yang menunjukkan satu fitur mendominasi secara ekstrem dibanding fitur lain merupakan sinyal tambahan yang layak dicurigai sebagai indikasi target leakage — fitur tersebut mungkin secara tidak sengaja menjadi "bocoran" langsung dari target.

## 1.8 Ringkasan Prinsip Operasional

* Data leakage secara formal adalah penggunaan informasi yang tidak akan tersedia pada saat prediksi sesungguhnya dibuat.
* Leakage bukan fenomena tunggal — terdapat minimal empat mekanisme berbeda (preprocessing, target, temporal, group), masing-masing dengan cara verifikasi yang berbeda pula.
* Aturan pencegahan universal: seluruh keputusan pemodelan harus dibuat seolah-olah test set "belum eksis" pada saat itu.
* Gejala utama leakage adalah kesenjangan performa yang tajam antara evaluasi internal (train/CV) dan performa pada data yang benar-benar baru.

# Bagian 2: Implementasi

## 2.1 Preprocessing Leakage — Efek dengan Ukuran Sampel Lebih Besar

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
np.random.seed(42)
n = 2000
X = pd.DataFrame({
    "fitur_1": np.random.randn(n) * 10 + 50,
    "fitur_2": np.random.randn(n) * 5 + 20,
})

In [3]:
y = pd.Series((X["fitur_1"] + X["fitur_2"] + np.random.randn(n) * 15 > 60).astype(int))

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


❌ SALAH: fit scaler ke seluruh X sebelum split (simulasi leakage)

In [7]:
scaler_leaks = StandardScaler().fit(X)
X_train_leaks = scaler_leaks.transform(X_train)
X_test_leaks = scaler_leaks.transform(X_test)

model_leaks = LogisticRegression().fit(X_train_leaks, y_train)
skor_leaks = accuracy_score(y_test, model_leaks.predict(X_test_leaks))

✅ BENAR: fit scaler hanya dilakukan di train

In [9]:
scaler_ok = StandardScaler().fit(X_train)
X_train_ok = scaler_ok.transform(X_train)
X_test_ok = scaler_ok.transform(X_test)

model_ok = LogisticRegression().fit(X_train_ok, y_train)
skor_ok = accuracy_score(y_test, model_ok.predict(X_test_ok))

In [10]:
print(f"Skor dengan leakage: {skor_leaks}")
print(f"Skor tanpa leakage: {skor_ok}")

Skor dengan leakage: 0.7575
Skor tanpa leakage: 0.7575


Catatan: pada dataset yang cukup besar dan homogen seperti contoh ini, selisih skornya bisa jadi kecil — ini bukan berarti leakage tidak berbahaya, melainkan menunjukkan bahwa besarnya dampak leakage bergantung pada karakteristik data. Pada dataset kecil, dengan outlier signifikan, atau dengan fitur berkardinalitas tinggi, dampaknya bisa jauh lebih besar (dibuktikan lebih jelas pada eksperimen 2.2 di bawah).

## 2.2 Target Leakage — Membuktikan Skor yang "Terlalu Bagus untuk Jadi Kenyataan"

In [11]:
np.random.seed(42)
n = 1000

# Simulasi kasus: prediksi "apakah nasabah gagal bayar (default)"
penghasilan = np.random.randn(n) * 2000000 + 6000000
umur = np.random.randint(20, 60, n)

# Target sesungguhnya: gagal bayar (1) atau tidak (0)
gagal_bayar = (penghasilan < 5000000).astype(int)

# FITUR BOCOR: "jumlah_tunggakan" hanya terisi > 0 JIKA gagal_bayar == 1
# Ini adalah informasi yang baru diketahui SETELAH target terjadi
jumlah_tunggakan = np.where(gagal_bayar == 1, np.random.randint(1, 5, n), 0)

X_leak = pd.DataFrame({
    "penghasilan": penghasilan,
    "umur": umur,
    "jumlah_tunggakan": jumlah_tunggakan   # fitur bermasalah
})
y_leak = pd.Series(gagal_bayar)

X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=42, stratify=y_leak
)

# Model DENGAN fitur bocor
model_bocor = LogisticRegression(max_iter=500).fit(X_train, y_train)
print("Akurasi DENGAN target leakage:", model_bocor.score(X_test, y_test))
print("Koefisien tiap fitur:", dict(zip(X_leak.columns, model_bocor.coef_[0])))

# Model TANPA fitur bocor (fitur "jumlah_tunggakan" dibuang)
X_train_bersih = X_train.drop(columns="jumlah_tunggakan")
X_test_bersih = X_test.drop(columns="jumlah_tunggakan")
model_bersih = LogisticRegression(max_iter=500).fit(X_train_bersih, y_train)
print("\nAkurasi TANPA target leakage:", model_bersih.score(X_test_bersih, y_test))

Akurasi DENGAN target leakage: 1.0
Koefisien tiap fitur: {'penghasilan': np.float64(-1.2305721531648068e-06), 'umur': np.float64(0.051073101512324144), 'jumlah_tunggakan': np.float64(5.1345199484336606)}

Akurasi TANPA target leakage: 0.805


perhatikan dua hal: (1) akurasi model dengan fitur bocor akan mendekati sempurna (mendekati 1.0), sesuai heuristik "terlalu bagus untuk jadi kenyataan" di bagian 1.7; (2) koefisien untuk jumlah_tunggakan akan bernilai jauh lebih besar dibanding fitur lain — bukti langsung dari sinyal tambahan yang dijelaskan di bagian 1.7 soal feature importance yang mendominasi secara ekstrem.

## 2.3 Temporal Leakage — Random Split vs Split Kronologis

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

np.random.seed(42)
n_hari = 500
tanggal = pd.date_range("2024-01-01", periods=n_hari, freq="D")

# Data punya TREN naik seiring waktu (pola umum di dunia nyata)
tren = np.linspace(100, 500, n_hari)
noise = np.random.randn(n_hari) * 20
penjualan = tren + noise

df_ts = pd.DataFrame({"tanggal": tanggal, "hari_ke": range(n_hari), "penjualan": penjualan})
df_ts["lag_1"] = df_ts["penjualan"].shift(1)
df_ts = df_ts.dropna()

X_ts = df_ts[["hari_ke", "lag_1"]]
y_ts = df_ts["penjualan"]

# ❌ SALAH: random split — baris "masa depan" bisa masuk ke train
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_ts, y_ts, test_size=0.2, random_state=42, shuffle=True
)
model_r = RandomForestRegressor(random_state=42).fit(X_train_r, y_train_r)
mae_random = mean_absolute_error(y_test_r, model_r.predict(X_test_r))

# ✅ BENAR: split kronologis — train = masa lalu, test = masa depan
split_point = int(len(df_ts) * 0.8)
X_train_k, X_test_k = X_ts.iloc[:split_point], X_ts.iloc[split_point:]
y_train_k, y_test_k = y_ts.iloc[:split_point], y_ts.iloc[split_point:]
model_k = RandomForestRegressor(random_state=42).fit(X_train_k, y_train_k)
mae_kronologis = mean_absolute_error(y_test_k, model_k.predict(X_test_k))

print("MAE dengan random split (temporal leakage):", mae_random)
print("MAE dengan split kronologis (benar)        :", mae_kronologis)

MAE dengan random split (temporal leakage): 18.996525391480464
MAE dengan split kronologis (benar)        : 35.53411501854785


MAE (error) pada skema random split biasanya lebih rendah/tampak lebih baik dibanding skema kronologis — ini karena model "mencontek" pola dari titik waktu di sekitar baris test (baik sebelum maupun sesudahnya), sesuatu yang tidak akan tersedia pada prediksi sungguhan di masa depan.

## 2.4 Group Leakage — Pembuktian dengan GroupShuffleSplit

In [14]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
n_pasien = 50
baris_per_pasien = 10

# Simulasi: setiap pasien punya beberapa baris pengukuran (mis. beberapa kunjungan)
pasien_id = np.repeat(range(n_pasien), baris_per_pasien)
fitur_dasar = np.random.randn(n_pasien).repeat(baris_per_pasien)  # karakteristik unik per pasien
noise_ukur = np.random.randn(n_pasien * baris_per_pasien) * 0.3

X_grp = pd.DataFrame({"pengukuran": fitur_dasar + noise_ukur})
y_grp = pd.Series((fitur_dasar + noise_ukur > 0).astype(int))
groups = pasien_id

# ❌ SALAH: random split biasa — baris dari pasien yang sama bisa "terpecah"
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_grp, y_grp, test_size=0.2, random_state=42
)
model_salah = RandomForestClassifier(random_state=42).fit(X_train_g, y_train_g)
skor_salah = model_salah.score(X_test_g, y_test_g)

# ✅ BENAR: GroupShuffleSplit — satu pasien HANYA masuk ke salah satu, train ATAU test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_grp, y_grp, groups=groups))
X_train_gg, X_test_gg = X_grp.iloc[train_idx], X_grp.iloc[test_idx]
y_train_gg, y_test_gg = y_grp.iloc[train_idx], y_grp.iloc[test_idx]

model_benar = RandomForestClassifier(random_state=42).fit(X_train_gg, y_train_gg)
skor_benar = model_benar.score(X_test_gg, y_test_gg)

# Verifikasi: pastikan tidak ada pasien yang sama di train dan test
pasien_train = set(groups[train_idx])
pasien_test = set(groups[test_idx])
print("Irisan pasien train & test (harus kosong):", pasien_train & pasien_test)

print("\nSkor TANPA group split (leakage) :", skor_salah)
print("Skor DENGAN GroupShuffleSplit     :", skor_benar)

Irisan pasien train & test (harus kosong): set()

Skor TANPA group split (leakage) : 1.0
Skor DENGAN GroupShuffleSplit     : 1.0


## 2.5 Prosedur Diagnosis: Membandingkan Skor Train vs Test sebagai Sinyal Awal

In [15]:
def audit_gap(model, X_train, y_train, X_test, y_test, nama=""):
    skor_train = model.score(X_train, y_train)
    skor_test = model.score(X_test, y_test)
    print(f"[{nama}] Skor train: {skor_train:.4f} | Skor test: {skor_test:.4f} "
          f"| Selisih: {skor_train - skor_test:.4f}")

audit_gap(model_bocor, X_train, y_train, X_test, y_test, "Dengan target leakage")
audit_gap(model_bersih, X_train_bersih, y_train, X_test_bersih, y_test, "Tanpa target leakage")

[Dengan target leakage] Skor train: 1.0000 | Skor test: 1.0000 | Selisih: 0.0000
[Tanpa target leakage] Skor train: 0.8275 | Skor test: 0.8050 | Selisih: 0.0225


Selisih yang sangat kecil dan skor yang mendekati sempurna pada keduanya (train maupun test) adalah pola karakteristik target leakage — berbeda dengan overfitting biasa, di mana skor train tinggi namun skor test jelas lebih rendah. Perbedaan pola ini penting untuk membedakan dua diagnosis yang berbeda akar penyebabnya.

# Kesalahan UMUM

| Kesalahan                                                                                  | Penjelasan & Dampak                                                                                                   |
| ------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------- |
| **Menganggap leakage hanya soal `fit_transform` yang salah tempat**                        | Target, temporal, dan group leakage bisa terjadi walau prosedur split/preprocessing sudah benar                       |
| **Tidak mencurigai skor yang sangat tinggi, justru merayakannya**                          | Skor “terlalu bagus” seharusnya memicu **audit leakage terlebih dahulu**, bukan langsung diklaim sebagai keberhasilan |
| **Memilih fitur berdasarkan kekuatan korelasi tanpa mengecek kronologi ketersediaannya**   | Fitur yang secara statistik sangat prediktif bisa jadi justru **turunan langsung dari target** (*target leakage*)     |
| **Melakukan random split pada data yang punya struktur pengelompokan (pasien, pelanggan)** | Terjadi **group leakage** — model tampak general, padahal hanya mengenali kembali entitas yang sudah dilihat          |
| **Hanya memeriksa gap skor train-test untuk mendeteksi masalah**                           | Target leakage bisa menghasilkan skor **train dan test sama-sama tinggi** — gap kecil bukan jaminan tidak ada masalah |


```text
Bagian 3: Latihan Praktik

Latihan 1 — Audit Taksonomi
Ambil dataset dari project/latihan pribadimu (atau dataset publik apa pun). Untuk setiap kolom fitur, tuliskan (dalam bentuk poin): apakah ada risiko target leakage (fitur baru tersedia setelah target diketahui)? Apakah ada risiko temporal leakage (data punya dimensi waktu)? Apakah ada risiko group leakage (ada entitas berulang)?

Latihan 2 — Perbesar Efek Preprocessing Leakage
Modifikasi kode 2.1 dengan memperkecil ukuran dataset (n=50 alih-alih 2000) dan menambahkan outlier ekstrem pada beberapa baris. Bandingkan selisih skor_leak vs skor_ok — apakah selisihnya menjadi lebih terlihat dibanding eksperimen dengan n=2000? Jelaskan kaitannya dengan ukuran sampel dan sensitivitas statistik terhadap outlier.

Latihan 3 — Deteksi Target Leakage Lewat Feature Importance
Ganti LogisticRegression pada kode 2.2 dengan RandomForestClassifier, lalu cetak model_bocor.feature_importances_. Apakah fitur jumlah_tunggakan mendominasi secara ekstrem dibanding fitur lain? Kaitkan hasil ini dengan sinyal kecurigaan di bagian 1.7.

Latihan 4 — Uji TimeSeriesSplit sebagai Alternatif
Cari dokumentasi sklearn.model_selection.TimeSeriesSplit. Terapkan pada dataset di kode 2.3 sebagai pengganti split manual berbasis split_point. Bandingkan hasilnya dengan pendekatan manual — apakah kesimpulannya (kronologis lebih valid dibanding random) tetap konsisten?

Latihan 5 — Group Leakage pada Skala Lebih Besar
Modifikasi kode 2.4 dengan memperbesar baris_per_pasien menjadi 30 (lebih banyak baris per entitas). Amati apakah selisih skor_salah vs skor_benar menjadi lebih besar. Jelaskan intuisimu: kenapa semakin banyak baris per entitas, potensi leakage-nya semakin besar jika tidak memakai GroupShuffleSplit?

Latihan 6 — Refleksi Tertulis (Sistematis)
Jawab dalam bentuk poin:

Definisikan data leakage sesuai definisi formal di bagian 1.1, dengan kata-kata sendiri
Sebutkan kelima jenis leakage dalam taksonomi, beserta satu contoh konkret untuk masing-masing (boleh berbeda dari contoh di materi ini)
Jelaskan perbedaan pola gejala antara target leakage vs overfitting biasa, dari sisi gap skor train-test
Jelaskan kenapa fitur yang "secara statistik sangat prediktif" tidak otomatis aman dipakai, tanpa mempertimbangkan kronologi data